# 11 Deep Agents Advanced

By the end you can wire backends, subagents, skills, memory, and HITL on durable /saved/ paths.


Set up the project path and reload shared helpers from this repo.


In [1]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\Azooo\langchain-bootcamp


Check which API keys and provider settings are available in `.env`.


In [2]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


DEEPSEEK_API_KEY: True
DEEPSEEK_MODEL: deepseek-v4-flash
LLM_PROVIDER: deepseek
TAVILY_API_KEY: True
LANGSMITH_API_KEY: True


Load the chat model helper used by live cells below.


In [4]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


Build specialists with `create_agent()`, wrap each graph in `CompiledSubAgent`, and wire `CompositeBackend` routes: `/workspace/` is ephemeral, `/saved/` is durable. Print lists subagent names, runnable types, and both path prefixes.


In [5]:

from deepagents import CompiledSubAgent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from shared.bootcamp_fixtures import check_weather, search_destinations

WORKSPACE_PATH = "/workspace/draft.txt"
SAVED_PATH = "/saved/itinerary.txt"


store = InMemoryStore()

# Ephemeral /workspace/* in graph state; durable /saved/* in the store.
travel_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/saved/": StoreBackend(
            store=store,
            namespace=lambda rt: ("travel", "saved"),
        ),
    },
)
model = require_llm()

weather_expert = create_agent(
    model=model,
    tools=[check_weather],
    system_prompt="You are a weather specialist. Return a concise forecast summary.",
)
deal_finder = create_agent(
    model=model,
    tools=[search_destinations],
    system_prompt="You find affordable destinations. Return the best match only.",
)

# CompiledSubAgent: pass create_agent() graphs as runnable, same API as notebook 03.
subagents = [
    CompiledSubAgent(
        name="weather_expert",
        description="Weather analysis and activity recommendations",
        runnable=weather_expert,
    ),
    CompiledSubAgent(
        name="deal_finder",
        description="Find destinations under the user's budget",
        runnable=deal_finder,
    ),
]

print("subagents:", [s["name"] for s in subagents])
print("runnable:", [type(s["runnable"]).__name__ for s in subagents])
print("ephemeral:", WORKSPACE_PATH)
print("durable:", SAVED_PATH)


subagents: ['weather_expert', 'deal_finder']
runnable: ['CompiledStateGraph', 'CompiledStateGraph']
ephemeral: /workspace/draft.txt
durable: /saved/itinerary.txt


Remote async subagents expose lifecycle tools for long-running workers. Print the five tool names, orchestrators delegate with task() while workers run asynchronously.


In [6]:
ASYNC_SUBAGENT_TOOLS = (
    "start_async_task",
    "check_async_task",
    "update_async_task",
    "cancel_async_task",
    "list_async_tasks",
)
print("remote async tools:", ASYNC_SUBAGENT_TOOLS)


remote async tools: ('start_async_task', 'check_async_task', 'update_async_task', 'cancel_async_task', 'list_async_tasks')


Define filesystem permission rules: allow workspace reads/writes, deny secrets, interrupt durable writes. Three rules print, interrupt on /saved/ is what triggers HITL later.


In [7]:
from deepagents.backends.protocol import SandboxBackendProtocol

FilesystemPermission = dict

permissions = [
    {"operations": ["read", "write"], "paths": ["/workspace/**"], "mode": "allow"},
    {"operations": ["read", "write"], "paths": [".env", "**/secrets/**"], "mode": "deny"},
    {"operations": ["write"], "paths": ["/saved/**"], "mode": "interrupt"},
]
print("permissions:", len(permissions))
print("execute requires:", SandboxBackendProtocol.__name__)
print("DATAFLOW (permissions): allow workspace, deny secrets, interrupt /saved/")


permissions: 3
execute requires: SandboxBackendProtocol
DATAFLOW (permissions): allow workspace, deny secrets, interrupt /saved/


Pass `CompiledSubAgent` specs into `create_deep_agent`. `task()` returns only the subagent final message to the parent, not every tool call (unlike shared-messages subgraphs). DATAFLOW should show `task` tool calls and a `write_file` path under `/saved/itinerary.txt`.


In [8]:

from deepagents import create_deep_agent
from shared.dataflow import print_agent_dataflow, print_rag_dataflow

ORCHESTRATOR_THREAD_CONFIG = {
    "configurable": {"thread_id": "nb11-orchestrator"},
    "recursion_limit": 50,
}

ORCHESTRATOR_PROMPT = """You orchestrate travel planning subagents.

Workflow (use write_todos — max 4 items):
1. task(deal_finder) once — beach destination under the user's budget.
2. task(weather_expert) once — forecast for that destination.
3. write_file — save ONE concise itinerary to /saved/itinerary.txt.
4. STOP — reply with a short summary; do not re-read files or call subagents again.

Efficiency rules:
- Call each subagent at most once with complete instructions.
- Do not call search_destinations or check_weather yourself; use task() only.
- After write_file succeeds, end immediately — no verification loops.
"""

orchestrator = create_deep_agent(
    model=model,
    subagents=subagents,
    backend=travel_backend,
    store=store,
    system_prompt=ORCHESTRATOR_PROMPT,
)

question = "Plan a beach trip under $600 with weather check; save to /saved/itinerary.txt"
result = orchestrator.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=ORCHESTRATOR_THREAD_CONFIG,
)

print_rag_dataflow(question, "task to write_file(/saved/itinerary.txt)")
print_agent_dataflow(result["messages"])


c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:13: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:14: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains store and context via `get_store()` / `get_runtime()`. Use `StoreBackend()` or `StoreBack

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

Call create_deep_agent with skills= and memory=, SKILL.md loads on demand, AGENTS.md injects every turn. Seed paths print; the agent reads them via filesystem tools when relevant.


In [ ]:

from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from shared.bootcamp_fixtures import TRAVEL_TOOLS

SKILL_MD = """---
name: visa-checklist
description: Schengen visa document checklist for EU travel
---
# Visa Checklist
- Passport valid 6+ months
- Travel insurance
- Return ticket
"""

AGENTS_MD = """# Travel Agent Memory
- Default currency: USD
- Mention visa requirements for non-EU destinations
- Prefer concise bullet summaries
"""

skill_agent = create_deep_agent(
    model=model,
    tools=TRAVEL_TOOLS,
    backend=travel_backend,
    store=store,
    skills=["/skills/"],
    memory=["/memory/AGENTS.md"],
    system_prompt="Use skills when visa rules apply. Follow AGENTS.md preferences.",
)

seed_files = {
    "/skills/visa-checklist/SKILL.md": create_file_data(SKILL_MD),
    "/memory/AGENTS.md": create_file_data(AGENTS_MD),
}

print("skills=", ["/skills/"])
print("memory=", ["/memory/AGENTS.md"])
print("seed:", list(seed_files))


skills= ['/skills/']
memory= ['/memory/AGENTS.md']
seed: ['/skills/visa-checklist/SKILL.md', '/memory/AGENTS.md']


Use FilesystemBackend for local dev, execute only works with SandboxBackendProtocol backends. Print confirms virtual_mode=True so paths stay sandboxed under the project root.


In [10]:
from deepagents.backends import FilesystemBackend

dev_backend = FilesystemBackend(root_dir=".", virtual_mode=True)
print("FilesystemBackend:", dev_backend.__class__.__name__, "virtual_mode=True")


FilesystemBackend: FilesystemBackend virtual_mode=True


create_deep_agent with interrupt_on write_file, stream until interrupt, approve via ask_approval, then resume with Command(resume={"decisions": [...]}).


In [11]:

from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from shared.bootcamp_fixtures import search_destinations
from shared.dataflow import preview, print_agent_dataflow
from shared.notebook_display import ask_approval

HITL_THREAD_CONFIG = {
    "configurable": {"thread_id": "nb11-hitl"},
    "recursion_limit": 50,
}

hitl_agent = create_deep_agent(
    model=model,
    tools=[search_destinations],
    backend=travel_backend,
    store=store,
    interrupt_on={"write_file": True},
    checkpointer=MemorySaver(),
    system_prompt=(
        "Plan the trip, write ONE summary to /saved/itinerary.txt, then stop. "
        "Do not re-read or rewrite the file after a successful write."
    ),
)

question = "Plan a Lisbon trip and write the summary to /saved/itinerary.txt"

for chunk in hitl_agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config=HITL_THREAD_CONFIG,
    stream_mode="updates",
):
    if "__interrupt__" in chunk:
        print("interrupt:", preview(chunk["__interrupt__"]))

state = hitl_agent.get_state(HITL_THREAD_CONFIG)
if state.interrupts:
    decision = "approve" if ask_approval(state.interrupts[0].value).get("approved") else "reject"
    result = hitl_agent.invoke(
        Command(resume={"decisions": [{"type": decision}]}),
        config=HITL_THREAD_CONFIG,
    )
    print_agent_dataflow(result["messages"])
else:
    print("No interrupt, model completed without write_file.")


c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:13: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:14: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains store and context via `get_store()` / `get_runtime()`. Use `StoreBackend()` or `StoreBack

interrupt: (Interrupt(value={'action_requests': [{'name': 'write_file', 'args': {'file_path': '/saved/itinerary.txt', 'content': "========================================\n         LISBON TRI...
HUMAN REVIEW
  action_requests: [{'name': 'write_file', 'args': {'file_path': '/saved/itinerary.txt', 'content': "========================================\n         LISBON TRIP ITINERARY\n========================================\n\nDestination : Lisbon, Portugal\nBudget      : $520 (per person)\nDuration    : 4 Days / 3 Nights (Suggested)\n\n========================================\nDAY 1 — ARRIVAL & ALFAMA EXPLORATION\n========================================\n- Arrive at Lisbon Portela Airport (LIS)\n- Check into accommodation (Alfama or Baixa district recommended)\n- Afternoon: Explore Alfama — the oldest district\n  - Wander narrow cobblestone streets\n  - Visit Lisbon Cathedral (Sé de Lisboa)\n  - Ride Tram 28 for a scenic tour\n- Evening: Fado dinner experience in a traditional Alfama res

c:\Users\Azooo\langchain-bootcamp\venv\Lib\site-packages\deepagents\middleware\filesystem.py:1679: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:13: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and writes state via `get_config()`. Use `StateBackend()` instead.
  default=StateBackend(rt),
C:\Users\Azooo\AppData\Local\Temp\ipykernel_27264\2572501386.py:14: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains store and context via `get_store()` / `get_runtime()`. Use `StoreBackend()` or `StoreBack

DATAFLOW
[0] HumanMessage: Plan a Lisbon trip and write the summary to /saved/itinerary.txt
[1] AIMessage tool_calls: ["search_destinations({'query': 'Lisbon city trip', 'budget': 2000})"]
[2] ToolMessage(search_destinations): Lisbon — $520
[3] AIMessage tool_calls: ["search_destinations({'query': 'Lisbon beach', 'budget': 2000})"]
[4] ToolMessage(search_destinations): Phuket — $450
[5] AIMessage tool_calls: ["search_destinations({'query': 'Lisbon Portugal city break', 'budget': 3000})"]
[6] ToolMessage(search_destinations): Lisbon — $520
[7] AIMessage tool_calls: ['write_file({\'file_path\': \'/saved/itinerary.txt\', \'content\': "========================================\\n         LISBON TRIP ITINERARY\\n========================================\\n\\nDestination ...)']
[8] ToolMessage(write_file): User rejected the tool call for `write_file` with id call_00_nH22fZCIWRTCx59hBWxY5588. The tool was not executed. Do not retry this tool call unless the user explicitly requests it...
[9] AI